# Revision experiments (Colab)
**Paper:** *Validation-Gated Causal Interventions for Interpreting High-Stakes LLM Behavior: A Case Study in Suicidality Detection*

This notebook runs every remaining analysis from `HEAD_LOCALIZATION_REVISION_RECOMMENDATIONS_2026-10-10.md`
(A1–A7) plus the corpus-overlap audit and the internal-monitor uncertainty, all in **fp16** on the
**pinned** checkpoint, with the same data draws as the manuscript scripts (verified by hash).

| § | What | Fixes | Needs GPU | ~Time (A100) |
|---|---|---|---|---|
| 1 | Data + tokenizer, hash check against the fp16 rerun | — | no | 2 min |
| 2 | Checkpoint provenance (which HF commits exist; are weights/tokenization identical?) | A7 | no | 2 min |
| 3 | Truncation counts for every sample in the paper | A6 | no | 2 min |
| 4 | Internal monitor: paired bootstrap + McNemar for the +11 TP claim | monitor CI | no | 1 min |
| 5 | Head-patching heatmap + split-half figure from the rerun | 3g figure | no | 1 min |
| 6 | Corpus overlap audit (exact + MinHash near-duplicates) | overlap | no | 10–15 min |
| 7 | Load model, reproduce direction ablation (environment check) | — | **yes** | 5 min |
| 8 | Direction ablation with **no truncation** | A6 | yes | 5 min |
| 9 | Per-head knockout on the held-out test set: AUC drop **and** LD-gap drop with CIs, fp16 power | A2, A5, A4 | yes | 45–60 min |
| 10 | **Joint** top-k head patching and knockout (cross-fitted) vs layer-matched random heads | A1 | yes | 15 min |
| 11 | Layer-wise attn/MLP patching, fp16, **unclipped**, 60 length-matched held-out pairs | A3, A4 | yes | 5 min |
| 12 | Causal + decoding transfer to SDCNL/SWMH in fp16, with and without overlapping posts | overlap, transfer numbers | yes | 10 min |
| 12b | Transfer after removing re-preprocessed duplicates of 232K posts (run §0–1, §7, §12, §12b only) | SDCNL provenance | yes | 10 min |
| 13 | Collect everything into `summary.json` + zip | — | no | — |

### Before you start
Uses the same Drive project as `llama_all_phases_colab.ipynb`: `MyDrive/MI on suicide/`.
1. Upload **`colab_data_bundle.zip`** (paper folder, `colab/`) to `MyDrive/MI on suicide/`
   (it holds 232K, the 174K "SDCNL" copy, SWMH and the fp16 rerun files, which are not in `Dataset/Inputs/`).
2. Colab ▸ 🔑 *Secrets*: add `HF_TOKEN` and enable notebook access. **Do not paste the token into a cell.**
3. *Runtime ▸ Change runtime type ▸* **A100** (L4 also works, ~3× slower; T4 does **not** fit fp16 8B).
4. *Runtime ▸ Run all*. The model loads from the Drive cache `MI on suicide/hf_cache/` (downloaded once if absent).
   Results land in `MI on suicide/Dataset/Outputs/revision_colab/`; §9 checkpoints there, so if Colab
   disconnects just re-run all — finished posts are skipped.

The UMD data are **not** in the bundle (DUA); nothing here needs them.

In [ ]:
# 0.1  Runtime + dependencies
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.56.2",
                    "accelerate>=0.33", "datasets>=2.19", "datasketch"], check=True)

In [ ]:
# 0.2  Paths, Drive, Hugging Face login, fixed settings
from pathlib import Path
import json, time, zipfile, copy
if IN_COLAB:
    from google.colab import drive, userdata
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    # Same Drive project as llama_all_phases_colab.ipynb
    DRIVE_PROJECT = Path("/content/drive/MyDrive/MI on suicide")
    assert DRIVE_PROJECT.exists(), f"{DRIVE_PROJECT} not found"
    BASE = DRIVE_PROJECT / "Dataset" / "Outputs"
    WORK = Path("/content/work")
    # Reuse the Llama weights already cached on Drive (no 16 GB re-download).
    # Must be set before transformers / huggingface_hub are imported.
    os.environ["HF_HOME"] = str(DRIVE_PROJECT / "hf_cache")
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(DRIVE_PROJECT / "hf_cache" / "hub")
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    BUNDLE_CANDIDATES = [DRIVE_PROJECT / "colab_data_bundle.zip",
                         DRIVE_PROJECT / "Dataset" / "Inputs" / "colab_data_bundle.zip",
                         DRIVE_PROJECT / "Dataset" / "colab_data_bundle.zip",
                         Path("/content/drive/MyDrive/colab_data_bundle.zip")]
else:
    BASE = Path(os.environ.get("REV_BASE", "."))
    WORK = Path(os.environ.get("REV_WORK", str(BASE / "work")))
    HF_TOKEN = os.environ.get("HF_TOKEN")
    BUNDLE_CANDIDATES = [BASE / "colab_data_bundle.zip"]

BUNDLE = next((p for p in BUNDLE_CANDIDATES if p.exists()), BUNDLE_CANDIDATES[0])
OUT = BASE / "revision_colab"
FIGS = OUT / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
if not (WORK / "data").exists():
    assert BUNDLE.exists(), "Upload colab_data_bundle.zip to one of: " + ", ".join(map(str, BUNDLE_CANDIDATES))
    print("unzipping bundle ..."); zipfile.ZipFile(BUNDLE).extractall(WORK)
DATA = WORK / "data" / "datasets"
RERUN = WORK / "results" / "sc2_binary"        # fp16 rerun at commit 0e9e39f2 (liu_headtohead_metallama_v2*)
OLD = WORK / "results" / "sc2_binary_fp16"     # the fp16 results the manuscript currently cites

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("No HF_TOKEN secret. Run:  from huggingface_hub import notebook_login; notebook_login()")

MODEL_ID = os.environ.get("REV_MODEL_ID", "meta-llama/Llama-3.1-8B-Instruct")
REVISION = os.environ.get("REV_REVISION", "0e9e39f249a16976918f6564b8830bc894c89659")
SEED = 0
LAYER = 16          # hidden_states[16] == output of decoder layer 15
ABLATE_FROM = 15    # direction ablation hooks decoder layers 15..31
MAXLEN = 256        # body capped at MAXLEN-80 = 176 tokens (manuscript protocol)
TEMPLATE = ("Read the following text and determine if the person shows signs of "
            "suicidal ideation.\n\n"
            "Text: {text}\n\n"
            "Does this text show signs of suicidal ideation? Answer with Yes or No.\n"
            "Answer:")
PREFIX, SUFFIX = TEMPLATE.split("{text}")
print("OUT ->", OUT)

In [ ]:
# 0.3  Shared helpers (hashing, AUC, bootstrap, BH, JSON output)
import hashlib, re
import numpy as np
from scipy import stats
from scipy.stats import rankdata

ENV = {"seed": SEED, "model": MODEL_ID, "revision_requested": REVISION}

def h16(t):
    """Same hash as sc2_liu_headtohead_v2.py."""
    return hashlib.sha256(" ".join(t.split()).encode()).hexdigest()[:16]

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1.0)

def auc_cols(y, S):
    """ROC-AUC of every column of S (ties -> average rank). y: 0/1 vector."""
    S = np.asarray(S, float)
    S = S[:, None] if S.ndim == 1 else S
    r = rankdata(S, axis=0)
    n1 = int(y.sum()); n0 = len(y) - n1
    out = (r[y == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)
    return out

def gap_cols(y, S):
    S = np.asarray(S, float); S = S[:, None] if S.ndim == 1 else S
    return S[y == 1].mean(0) - S[y == 0].mean(0)

def strat_boot(y, B=2000, seed=SEED + 7):
    rng = np.random.default_rng(seed)
    P, N = np.where(y == 1)[0], np.where(y == 0)[0]
    for _ in range(B):
        yield np.concatenate([rng.choice(P, len(P)), rng.choice(N, len(N))])

def boot_p(d):
    """Two-sided bootstrap p-value for H0: effect = 0 (rows = resamples)."""
    return np.minimum(1.0, 2 * np.minimum((d <= 0).mean(0), (d >= 0).mean(0)))

def _js(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def save_json(name, obj):
    path = OUT / name
    path.write_text(json.dumps({**obj, "env": ENV}, indent=2, default=_js))
    print("saved", path)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"

## 1. Data and tokenizer (CPU)
Exactly the draws used by the manuscript scripts: filter `len(text) > 80`, shuffle with `default_rng(0)`.
The test set is checked hash-for-hash against the fp16 rerun's per-example CSV.

In [ ]:
# 1.1  Suicide-Watch-232K draw (sc2_faithfulness.py / sc2_liu_headtohead_v2.py)
from datasets import load_from_disk
import csv
ds07 = load_from_disk(str(DATA / "07_Suicide_Watch_232K/hf_cache/train"))
T07, L07 = ds07["text"], ds07["label"]

def draw_main(seed=SEED):
    rng = np.random.default_rng(seed)
    sui = [t for t, l in zip(T07, L07) if l == "suicide" and t and len(t) > 80]
    non = [t for t, l in zip(T07, L07) if l == "non-suicide" and t and len(t) > 80]
    rng.shuffle(sui); rng.shuffle(non)
    return sui, non, rng

SUI, NON, _rng = draw_main()
RNG_AFTER_SHUFFLE = copy.deepcopy(_rng)   # the random-control direction is the next draw from this
SUI_TR, NON_TR = SUI[:100], NON[:100]          # direction fit
SUI_TE, NON_TE = SUI[100:300], NON[100:300]    # held-out test; first 100/class == manuscript test set

abl_rows = list(csv.DictReader(open(RERUN / "liu_headtohead_metallama_v2_ablation_examples.csv")))
assert [r["hash"] for r in abl_rows] == [h16(t) for t in SUI_TE + NON_TE], "test draw differs from the rerun!"
print(f"pools: {len(SUI)} suicide / {len(NON)} non-suicide;  test set matches the fp16 rerun (400/400 hashes)")

In [ ]:
# 1.2  External corpora: exact rng sequence of sc2_crossablation.py / sc2_generalization.py / sc2_internal_monitor.py
dssd = load_from_disk(str(DATA / "08_SDCNL/hf_cache/train"))
print("columns  232K:", ds07.column_names, "| 'SDCNL':", dssd.column_names, f"({len(dssd)} rows)")

def load_swmh(split):
    with open(DATA / "01_SWMH_Reddit" / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f]
SWMH_TRAIN = load_swmh("train")
print("SWMH columns:", sorted(SWMH_TRAIN[0].keys()), f"({len(SWMH_TRAIN)} train rows)")

def draw_external(n_ext=500):
    """The 80/class (cross-ablation) and 120/class (generalization) samples are prefixes of this draw."""
    rng = np.random.default_rng(SEED)
    def clean(texts, n):
        out = [t for t in texts if t and len(t) > 80]; rng.shuffle(out); return out[:n]
    s07 = clean([t for t, l in zip(T07, L07) if l == "suicide"], 100)
    n07 = clean([t for t, l in zip(T07, L07) if l == "non-suicide"], 100)
    sd = (clean([t for t, c in zip(dssd["text"], dssd["class"]) if c == "suicide"], n_ext),
          clean([t for t, c in zip(dssd["text"], dssd["class"]) if c == "non-suicide"], n_ext))
    sw = (clean([r.get("text", "") for r in SWMH_TRAIN if r.get("label", "") == "self.SuicideWatch"], n_ext),
          clean([r.get("text", "") for r in SWMH_TRAIN
                 if r.get("label", "") not in ("self.SuicideWatch", "self.depression")], n_ext))
    return {"s07": s07, "n07": n07, "SDCNL": sd, "SWMH": sw}, rng

EXT, RNG_AFTER_EXT = draw_external(500)
assert EXT["s07"] == SUI_TR and EXT["n07"] == NON_TR
print("external draws ok; 232K direction-train set identical across scripts")

In [ ]:
# 1.3  Tokenizer + prompt builder (identical to the manuscript scripts)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
YES = tok.encode(" Yes", add_special_tokens=False)[0]
NO = tok.encode(" No", add_special_tokens=False)[0]

def chat_ids(text, maxlen=MAXLEN):
    body = tok(text, add_special_tokens=False)["input_ids"]
    truncated = len(body) > maxlen - 80
    content = PREFIX + tok.decode(body[: maxlen - 80]) + SUFFIX
    s = tok.apply_chat_template([{"role": "user", "content": content}],
                                add_generation_prompt=True, tokenize=False)
    return tok(s, add_special_tokens=False, return_tensors="pt")["input_ids"], truncated

def body_len(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

ids0, tr0 = chat_ids(SUI_TE[0])
assert ids0.shape[1] == int(abl_rows[0]["n_tokens"]), "prompt length differs from the rerun"
print("prompt builder matches the rerun (token count check)")

## 2. Checkpoint provenance (A7, CPU)
The original fp16 runs did not record a commit. This lists every commit of the model repo and checks whether
the **weights** (safetensors LFS hashes) or the **tokenizer/chat template** ever changed, then tokenizes 50
real prompts under every distinct tokenizer revision. If weights and tokenization are identical across all
revisions, the results cannot depend on which revision was used, which closes the provenance gap.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

def _sha(f):
    lfs = getattr(f, "lfs", None)
    if lfs is None: return getattr(f, "blob_id", None)
    return getattr(lfs, "sha256", None) or (lfs.get("sha256") if isinstance(lfs, dict) else None)

prov_rows = []
for c in api.list_repo_commits(MODEL_ID):
    files = {f.path: f for f in api.list_repo_tree(MODEL_ID, revision=c.commit_id, recursive=True)
             if hasattr(f, "blob_id")}
    w = sorted((p, _sha(f)) for p, f in files.items() if p.endswith(".safetensors") and "/" not in p)
    blob = lambda p: files[p].blob_id if p in files else None
    prov_rows.append({"commit": c.commit_id, "date": str(c.created_at)[:10], "title": c.title,
                      "weights": hashlib.sha256(repr(w).encode()).hexdigest()[:12] if w else None,
                      "tokenizer_config": blob("tokenizer_config.json"), "tokenizer_json": blob("tokenizer.json"),
                      "config": blob("config.json"), "generation_config": blob("generation_config.json")})
for r in prov_rows:
    print(f"{r['date']}  {r['commit'][:8]}  weights={r['weights']}  tok_cfg={str(r['tokenizer_config'])[:8]}  "
          f"tok_json={str(r['tokenizer_json'])[:8]}  {r['title'][:50]}")

weight_sets = {r["weights"] for r in prov_rows if r["weights"]}
tok_revs = {}
for r in prov_rows:
    if r["tokenizer_config"] and r["tokenizer_json"]:
        tok_revs.setdefault((r["tokenizer_config"], r["tokenizer_json"]), r["commit"])
probe_texts = SUI_TE[:25] + NON_TE[:25]
ref_ids = [chat_ids(t)[0].tolist() for t in probe_texts]
tok_identical = {}
for key, commit in tok_revs.items():
    tk = AutoTokenizer.from_pretrained(MODEL_ID, revision=commit)
    same = True
    for t, ref in zip(probe_texts, ref_ids):
        body = tk(t, add_special_tokens=False)["input_ids"]
        content = PREFIX + tk.decode(body[: MAXLEN - 80]) + SUFFIX
        s = tk.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, tokenize=False)
        same &= tk(s, add_special_tokens=False)["input_ids"] == ref[0]
    tok_identical[commit] = bool(same)
print("\ndistinct weight sets:", len(weight_sets))
print("tokenization identical to pinned revision, per distinct tokenizer revision:", tok_identical)
PROVENANCE = {"commits": prov_rows, "n_distinct_weight_sets": len(weight_sets),
              "tokenization_identical_by_tokenizer_revision": tok_identical,
              "revision_independent": len(weight_sets) == 1 and all(tok_identical.values())}
print("=> results are revision-independent:", PROVENANCE["revision_independent"])
save_json("provenance.json", PROVENANCE)

## 3. Truncation counts (A6, CPU)
The protocol caps the post body at `MAXLEN-80` tokens (176 for `MAXLEN=256`; 120 for the edge-attribution
script, which used `maxlen=200`). This counts truncated posts per class for every sample the paper uses, and
flags samples that are subsets of the direction-training set (they are not held out).

In [ ]:
samples = {
    "direction_train (100/class)":                    (SUI_TR, NON_TR, 256),
    "ablation_test_manuscript (100/class)":           (SUI_TE[:100], NON_TE[:100], 256),
    "ablation_test_rerun (200/class)":                (SUI_TE, NON_TE, 256),
    "head_knockout_fp16 (sc2_head_knockout --n 60)":  (SUI[:60], NON[:60], 256),
    "layer_patching_fp16 (sc2_patching --n 20)":      (SUI[:20], NON[:20], 256),
    "edge_attribution_fp16 (n=30, maxlen=200)":       (SUI[:30], NON[:30], 200),
    "internal_monitor_eval (1000/class)":             (SUI[700:1700], NON[700:1700], 256),
    "crossablation SDCNL (80/class)":                 (EXT["SDCNL"][0][:80], EXT["SDCNL"][1][:80], 256),
    "crossablation SWMH (80/class)":                  (EXT["SWMH"][0][:80], EXT["SWMH"][1][:80], 256),
    "generalization SDCNL (120/class)":               (EXT["SDCNL"][0][:120], EXT["SDCNL"][1][:120], 256),
    "generalization SWMH (120/class)":                (EXT["SWMH"][0][:120], EXT["SWMH"][1][:120], 256),
}
train_h = {h16(t) for t in SUI_TR + NON_TR}
TRUNC = {}
print(f"{'sample':50s} {'cap':>4s} {'trunc sui':>10s} {'trunc non':>10s}  subset-of-train")
for name, (s, n, ml) in samples.items():
    cap = ml - 80
    ts = sum(body_len(t) > cap for t in s); tn = sum(body_len(t) > cap for t in n)
    sub = all(h16(t) in train_h for t in s + n)
    TRUNC[name] = {"body_cap_tokens": cap, "n_per_class": [len(s), len(n)],
                   "truncated_suicide": ts, "truncated_non_suicide": tn, "subset_of_direction_train": sub}
    print(f"{name:50s} {cap:4d} {ts:4d}/{len(s):<5d} {tn:4d}/{len(n):<5d}  {sub}")
save_json("truncation_counts.json", TRUNC)

## 4. Internal monitor: paired uncertainty (CPU)
The fp16 `internal_monitor.json` already stores all 2,000 held-out scores. At the model's alarm count (839),
the probe flags its top-839 posts; we bootstrap the recall difference (re-selecting top-K inside every
resample) and run an exact McNemar test on the discordant suicidal posts.

In [ ]:
mon = json.load(open(OLD / "internal_monitor.json"))
e = mon["eval_scores"]; ym = np.array(e["y"]); ldm = np.array(e["behavioral_ld"]); pm = np.array(e["probe"])
thr_b = mon["thresholds"]["behav"]

def topk_mask(s, k):
    f = np.zeros(len(s), bool); f[np.argsort(-s, kind="stable")[:k]] = True; return f

bflag = ldm > thr_b; K = int(bflag.sum()); pflag = topk_mask(pm, K); pos = ym == 1
tp_b, tp_p = int((bflag & pos).sum()), int((pflag & pos).sum())
p_only, b_only = int((pflag & ~bflag & pos).sum()), int((bflag & ~pflag & pos).sum())
mcnemar_p = stats.binomtest(p_only, p_only + b_only, 0.5).pvalue
diffs = []
for idx in strat_boot(ym, 10000, SEED):
    bf = ldm[idx] > thr_b; pf = topk_mask(pm[idx], int(bf.sum())); yy = ym[idx] == 1
    diffs.append(((pf & yy).sum() - (bf & yy).sum()) / yy.sum())
diffs = np.array(diffs); lo, hi = ci(diffs)
MONITOR = {"alarm_count": K, "tp_model": tp_b, "tp_probe": tp_p, "net_tp": tp_p - tp_b,
           "recall_diff": (tp_p - tp_b) / pos.sum(), "recall_diff_ci95": [lo, hi],
           "discordant_probe_only": p_only, "discordant_model_only": b_only, "mcnemar_exact_p": mcnemar_p}
print(f"matched alarm count {K}: probe TP {tp_p} vs model TP {tp_b} (net {tp_p - tp_b:+d})")
print(f"recall difference {MONITOR['recall_diff']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}];  "
      f"McNemar exact p = {mcnemar_p:.3f} ({p_only} vs {b_only} discordant)")
save_json("internal_monitor_paired.json", MONITOR)

## 5. Head-patching figures from the fp16 rerun (CPU)
Heatmap of signed, unclipped mean IE over the 60 primary pairs, plus the split-half scatter (recommendation 3g).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

npz = np.load(RERUN / "liu_headtohead_metallama_v2_ie.npz")
IE_ALL, PRIM = npz["ie"], npz["primary"].astype(bool)
IEp = IE_ALL[PRIM]; nP = len(IEp); NLy, NHy = IEp.shape[1:]
mean_ie = IEp.mean(0)
bidx = np.random.default_rng(SEED + 11).integers(0, nP, size=(2000, nP))
boot_mean = np.stack([IEp[b].mean(0) for b in bidx])
lo_ie, hi_ie = np.percentile(boot_mean, 2.5, 0), np.percentile(boot_mean, 97.5, 0)
perm = np.random.default_rng(SEED + 13).permutation(nP)
hA, hB = IEp[perm[: nP // 2]].mean(0).ravel(), IEp[perm[nP // 2:]].mean(0).ravel()
rho_split = stats.spearmanr(hA, hB)[0]
order = np.argsort(-mean_ie.ravel())
print("top heads by mean IE (60 held-out pairs):")
for i in order[:10]:
    L, h = divmod(int(i), NHy)
    print(f"  L{L}H{h}: {fmt_ci(mean_ie[L, h], lo_ie[L, h], hi_ie[L, h])}")
print(f"split-half Spearman over all heads: {rho_split:.2f}; top-20 overlap "
      f"{len(set(np.argsort(-hA)[:20]) & set(np.argsort(-hB)[:20]))}/20")

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.35, 1]})
v = np.abs(mean_ie).max()
im = ax[0].imshow(mean_ie, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
ax[0].set_xlabel("head"); ax[0].set_ylabel("layer"); ax[0].set_title("a  Mean IE per head (60 pairs, signed)", loc="left")
fig.colorbar(im, ax=ax[0], fraction=0.046, label="mean IE")
ax[1].scatter(hA, hB, s=6, c="0.3", alpha=0.6, lw=0)
m = max(np.abs(hA).max(), np.abs(hB).max()); ax[1].plot([-m, m], [-m, m], c="0.7", lw=0.8, ls="--")
ax[1].set_xlabel("mean IE, half A (30 pairs)"); ax[1].set_ylabel("mean IE, half B (30 pairs)")
ax[1].set_title(f"b  Split-half stability (Spearman {rho_split:.2f})", loc="left")
fig.tight_layout(); fig.savefig(FIGS / "head_patching_ie.png", dpi=200); fig.savefig(FIGS / "head_patching_ie.pdf")
plt.show()

## 6. Corpus overlap audit (CPU)
Normalised text (lower-case, URLs removed, punctuation collapsed). Exact duplicates by hash; near-duplicates by
MinHash-LSH on word 5-shingles (64 permutations, LSH threshold 0.8), confirmed with exact Jaccard ≥ 0.8.
Also checks the actual evaluation samples against the 232K direction-training posts.
None of the three corpora ships author IDs (see column lists in §1.2), so a **user**-disjoint check is not possible;
say so in the paper.

In [ ]:
from datasketch import MinHash, MinHashLSH

def norm(t):
    t = (t or "").lower(); t = re.sub(r"https?://\S+", " ", t); t = re.sub(r"[^a-z0-9']+", " ", t)
    return " ".join(t.split())

def nhash(t): return hashlib.md5(norm(t).encode()).hexdigest()

def shingles(t, k=5):
    w = norm(t).split()
    return {" ".join(w[i:i + k]) for i in range(max(1, len(w) - k + 1))} if w else set()

def jacc(a, b): return len(a & b) / max(1, len(a | b))

SWMH_ALL = [r for s in ("train", "validation", "test") for r in load_swmh(s)]
corpora = {"232K": (list(T07), list(L07)),
           "SDCNL": (list(dssd["text"]), list(dssd["class"])),
           "SWMH": ([r.get("text", "") for r in SWMH_ALL], [r.get("label", "") for r in SWMH_ALL])}

H232 = {}
for t, l in zip(*corpora["232K"]): H232.setdefault(nhash(t), l)
OVERLAP = {"exact": {}, "near": {}, "eval_samples": {}}
for name in ("SDCNL", "SWMH"):
    texts, labs = corpora[name]
    hits = [(nhash(t) in H232) for t in texts]
    within = len(texts) - len({nhash(t) for t in texts})
    OVERLAP["exact"][name] = {"n": len(texts), "exact_in_232K": int(sum(hits)), "within_corpus_duplicates": within}
    print(f"{name}: {sum(hits)}/{len(texts)} exact duplicates of 232K posts; {within} within-corpus duplicates")
print(f"232K within-corpus duplicates: {len(T07) - len(H232)}")

t0 = time.time()
lsh = MinHashLSH(threshold=0.8, num_perm=64)
SH232 = {}
for i, t in enumerate(T07):
    sh = shingles(t)
    if len(sh) < 3: continue
    mh = MinHash(num_perm=64, seed=1); mh.update_batch([s.encode() for s in sh])
    lsh.insert(str(i), mh)
    if i % 50000 == 0: print(f"  indexed {i}/{len(T07)}  ({time.time() - t0:.0f}s)")

def near_dups_232k(text):
    sh = shingles(text)
    if len(sh) < 3: return []
    mh = MinHash(num_perm=64, seed=1); mh.update_batch([s.encode() for s in sh])
    return [int(k) for k in lsh.query(mh) if jacc(sh, shingles(T07[int(k)])) >= 0.8]

for name in ("SDCNL", "SWMH"):
    texts, _ = corpora[name]
    # full-corpus rate on a 20k random subsample (full SDCNL query is slow); eval samples below are exhaustive
    sub = np.random.default_rng(SEED).choice(len(texts), min(20000, len(texts)), replace=False)
    nd = sum(bool(near_dups_232k(texts[i])) for i in sub)
    OVERLAP["near"][name] = {"n_checked": len(sub), "near_dup_in_232K": nd}
    print(f"{name}: {nd}/{len(sub)} sampled posts have a near-duplicate (J>=0.8) in 232K  ({time.time() - t0:.0f}s)")

train_sh = [shingles(t) for t in SUI_TR + NON_TR]
FLAGGED = set()   # h16 of eval posts that overlap 232K (used in §12)
for name in ("SDCNL", "SWMH"):
    for cls, texts in zip(("suicide", "non_suicide"), EXT[name]):
        ex = [nhash(t) in H232 for t in texts]
        nr = [bool(near_dups_232k(t)) for t in texts]
        tr = [max(jacc(shingles(t), s) for s in train_sh) >= 0.8 for t in texts]
        for t, a, b in zip(texts, ex, nr):
            if a or b: FLAGGED.add(h16(t))
        OVERLAP["eval_samples"][f"{name}/{cls}"] = {
            "n_500": len(texts),
            **{f"first{k}": {"exact_232K": int(sum(ex[:k])), "near_232K": int(sum(nr[:k])),
                             "near_direction_train": int(sum(tr[:k]))} for k in (80, 120, 500)}}
        print(f"{name}/{cls}: " + "; ".join(
            f"first {k}: exact {sum(ex[:k])}, near {sum(nr[:k])}, ~train {sum(tr[:k])}" for k in (80, 120, 500)))
OVERLAP["flagged_eval_hashes"] = sorted(FLAGGED)
save_json("corpus_overlap.json", OVERLAP)

## 7. Load the model (GPU) and reproduce the direction ablation
Pinned commit, fp16, single GPU. Fits the difference-of-means direction on the 100/class training posts and
reproduces baseline / ablated / random AUC on the manuscript test set; per-example logit differences are compared
with the rerun CSV as an environment check (expect agreement to within ~0.1 logits and AUC to within 0.005).

In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM
DEV = "cuda"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float16,
                                             device_map={"": 0})
model.eval()
LAYERS = model.model.layers
NL, NH = len(LAYERS), model.config.num_attention_heads
HD = getattr(model.config, "head_dim", None) or model.config.hidden_size // NH
DM = model.config.hidden_size
ENV.update(revision_resolved=getattr(model.config, "_commit_hash", None), precision="fp16",
           transformers=transformers.__version__, torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
print(ENV)

@torch.inference_mode()
def ld_batch(ids):
    lg = model(ids.to(DEV), logits_to_keep=1).logits[:, -1, :].float()
    return (lg[:, YES] - lg[:, NO]).cpu().numpy()

@torch.inference_mode()
def hidden_states(ids):
    out = model(ids.to(DEV), output_hidden_states=True, logits_to_keep=1)
    return torch.stack([h[0, -1] for h in out.hidden_states]).float().cpu().numpy()   # [NL+1, DM]

def make_ablate_hook(vec):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        v = vec.to(h.dtype)
        h2 = h - (h @ v).unsqueeze(-1) * v
        return (h2,) + tuple(out[1:]) if isinstance(out, tuple) else h2
    return hook

def score(ids_list, vec=None, layers=None):
    """LD per prompt, optionally ablating `vec` (np array) at decoder layers `layers` (default 15..31)."""
    handles = []
    if vec is not None:
        layers = range(ABLATE_FROM, NL) if layers is None else layers
        v = torch.tensor(vec, device=DEV, dtype=torch.float16)
        handles = [LAYERS[i].register_forward_hook(make_ablate_hook(v)) for i in layers]
    try:
        return np.array([ld_batch(i)[0] for i in ids_list])
    finally:
        for x in handles: x.remove()

# direction: difference of means at hidden_states[16] over the training posts
H_TR = np.stack([hidden_states(chat_ids(t)[0]) for t in SUI_TR + NON_TR])     # [200, NL+1, DM]
DIR = H_TR[:100, LAYER].mean(0) - H_TR[100:, LAYER].mean(0); DIR /= np.linalg.norm(DIR)
_r = copy.deepcopy(RNG_AFTER_SHUFFLE)
RAND0 = _r.standard_normal(DM); RAND0 /= np.linalg.norm(RAND0)                 # same draw as sc2_faithfulness.py
np.save(OUT / "direction_L16.npy", DIR)

TEST_TEXTS = SUI_TE + NON_TE
Y_TEST = np.array([1] * len(SUI_TE) + [0] * len(NON_TE))
TEST_IDS = [chat_ids(t)[0] for t in TEST_TEXTS]
m100 = np.r_[np.arange(100), 200 + np.arange(100)]
sc7 = {"baseline": score([TEST_IDS[i] for i in m100]),
       "direction": score([TEST_IDS[i] for i in m100], DIR),
       "random0": score([TEST_IDS[i] for i in m100], RAND0)}
y100 = Y_TEST[m100]
ref = {k: np.array([float(abl_rows[i][c]) for i in m100])
       for k, c in (("baseline", "baseline"), ("direction", "direction_ablation"), ("random0", "random_0"))}
for k in sc7:
    print(f"{k:10s} AUC {auc_cols(y100, sc7[k])[0]:.4f} (rerun {auc_cols(y100, ref[k])[0]:.4f}; manuscript "
          f"{ {'baseline': 0.975, 'direction': 0.832, 'random0': 0.975}[k] })   "
          f"max |LD - rerun LD| = {np.abs(sc7[k] - ref[k]).max():.3f}")

## 8. Direction ablation without truncation (A6 sensitivity)
Same direction (fit under the standard protocol), but test prompts carry the **full** post body.

In [ ]:
TEST_IDS_FULL = [chat_ids(t, maxlen=4096)[0] for t in TEST_TEXTS]
trunc_mask = np.array([body_len(t) > MAXLEN - 80 for t in TEST_TEXTS])
sc8 = {k: score(TEST_IDS_FULL, v) for k, v in (("baseline", None), ("direction", DIR), ("random0", RAND0))}
NOTRUNC = {}
for tag, mask in (("manuscript_100pc", np.isin(np.arange(400), m100)), ("rerun_200pc", np.ones(400, bool))):
    y = Y_TEST[mask]; S = np.stack([sc8[k][mask] for k in ("baseline", "direction", "random0")], 1)
    a = auc_cols(y, S); bt = np.stack([auc_cols(y[i], S[i]) for i in strat_boot(y)])
    drop, drop_r = bt[:, 0] - bt[:, 1], bt[:, 0] - bt[:, 2]
    NOTRUNC[tag] = {"n_truncated_under_protocol": int(trunc_mask[mask].sum()),
                    "auc": dict(zip(("baseline", "direction", "random0"), a)),
                    "drop_direction": a[0] - a[1], "drop_direction_ci95": ci(drop),
                    "drop_random": a[0] - a[2], "drop_random_ci95": ci(drop_r)}
    print(f"{tag}: no truncation -> baseline {a[0]:.3f}, direction {a[1]:.3f}, random {a[2]:.3f}; "
          f"drop {fmt_ci(a[0] - a[1], *ci(drop))}")
save_json("ablation_no_truncation.json", NOTRUNC)

## 9. Per-head knockout on the held-out test set (A2, A5, A4)
Zero one head's slice of the `o_proj` input at all positions (as in `sc2_head_knockout.py`), on the **held-out**
test posts that also carry the direction-ablation result, so both estimands are measured on the same posts.
For every post and layer, one batched forward runs 32 knockout rows plus one unpatched row (the per-batch
baseline). Reports, per head: **AUC drop** and **LD-gap drop**, each with a stratified paired bootstrap CI,
BH-corrected bootstrap p-values, and an fp16 power analysis.

`N_KO = 100` (the manuscript test set, ~45 min on A100) or `200` (the whole rerun test set, ~90 min).
Progress is checkpointed to Drive every 10 posts.

In [ ]:
N_KO = 100
ko_idx = np.r_[np.arange(N_KO), 200 + np.arange(N_KO)]
Y_KO = Y_TEST[ko_idx]
ckpt = OUT / f"knockout_n{N_KO}_partial.npz"
if ckpt.exists():
    z = np.load(ckpt); LDKO, BASEKO = z["ldko"], z["base"]
else:
    LDKO = np.full((len(ko_idx), NL, NH), np.nan, np.float32); BASEKO = np.full((len(ko_idx), NL), np.nan, np.float32)
KEEP = torch.ones(NH + 1, 1, NH * HD, device=DEV, dtype=torch.float16)
for h in range(NH): KEEP[h, :, h * HD:(h + 1) * HD] = 0
def ko_prehook(module, inp): return (inp[0] * KEEP,) + tuple(inp[1:])

t0 = time.time(); done0 = int((~np.isnan(BASEKO[:, -1])).sum())
for j, i in enumerate(ko_idx):
    if not np.isnan(BASEKO[j, -1]): continue
    ids = TEST_IDS[i].repeat(NH + 1, 1)
    for L in range(NL):
        hk = LAYERS[L].self_attn.o_proj.register_forward_pre_hook(ko_prehook)
        try:
            out = ld_batch(ids)
        finally:
            hk.remove()
        LDKO[j, L], BASEKO[j, L] = out[:NH], out[NH]
    n_done = int((~np.isnan(BASEKO[:, -1])).sum())
    if n_done % 10 == 0 or n_done == len(ko_idx):
        np.savez(ckpt, ldko=LDKO, base=BASEKO)
        rate = (time.time() - t0) / max(1, n_done - done0)
        print(f"  {n_done}/{len(ko_idx)} posts  ({rate:.1f}s/post, ~{rate * (len(ko_idx) - n_done) / 60:.0f} min left)")
np.savez(OUT / f"knockout_n{N_KO}_per_example.npz", ldko=LDKO, base=BASEKO, y=Y_KO,
         hashes=np.array([h16(TEST_TEXTS[i]) for i in ko_idx]))

In [ ]:
# 9b. Knockout statistics + fp16 power + figure (CPU)
base_ld = BASEKO.mean(1)
print(f"unpatched-row consistency across layers: max spread {np.nanmax(BASEKO.max(1) - BASEKO.min(1)):.4f} logits")
S = np.concatenate([base_ld[:, None], LDKO.reshape(len(ko_idx), -1)], 1)          # [n, 1+1024]
a_all, g_all = auc_cols(Y_KO, S), gap_cols(Y_KO, S)
auc_drop, gap_drop = a_all[0] - a_all[1:], g_all[0] - g_all[1:]
BA, BG = [], []
for idx in strat_boot(Y_KO, 2000):
    a, g = auc_cols(Y_KO[idx], S[idx]), gap_cols(Y_KO[idx], S[idx])
    BA.append(a[0] - a[1:]); BG.append(g[0] - g[1:])
BA, BG = np.array(BA), np.array(BG)
lo_a, hi_a = ci(BA); lo_g, hi_g = ci(BG)
q_a, q_g = bh(boot_p(BA)), bh(boot_p(BG))
se_a = BA.std(0)
mde = (1.959964 + 0.841621) * np.median(se_a)
top = int(np.argmax(auc_drop)); topg = int(np.argmax(gap_drop))
hn = lambda k: f"L{k // NH}H{k % NH}"
# Hanley-McNeil conservative MDE, for comparability with the old (4-bit) head_knockout_power.json
A0 = a_all[0]; n1 = N_KO
q1, q2 = A0 / (2 - A0), 2 * A0 * A0 / (1 + A0)
se_hm = np.sqrt((A0 * (1 - A0) + (n1 - 1) * (q1 - A0 ** 2) + (n1 - 1) * (q2 - A0 ** 2)) / (n1 * n1))
mde_hm = (1.959964 + 0.841621) * np.sqrt(2) * se_hm

print(f"baseline AUC {A0:.4f}, LD gap {g_all[0]:.2f}  (n={N_KO}/class, held out)")
print(f"largest AUC drop: {hn(top)} {fmt_ci(auc_drop[top], lo_a[top], hi_a[top])}; heads with BH q<0.05: "
      f"{int((q_a < 0.05).sum())}/1024; mean drop {auc_drop.mean():+.4f}")
print(f"largest LD-gap drop: {hn(topg)} {fmt_ci(gap_drop[topg], lo_g[topg], hi_g[topg], 2)} of {g_all[0]:.2f}; "
      f"heads with BH q<0.05: {int((q_g < 0.05).sum())}/1024")
print(f"fp16 power: median paired SE(AUC drop) {np.median(se_a):.4f} -> MDE(80%) {mde:.4f}; "
      f"max per-head upper 95% bound {hi_a.max():.4f}; Hanley-McNeil conservative MDE {mde_hm:.4f}")
for name in ("L15H4", "L10H27", "L9H19", "L12H4", "L14H24", "L9H17", "L11H1"):
    L, h = map(int, name[1:].split("H")); k = L * NH + h
    print(f"  {name}: AUC drop {fmt_ci(auc_drop[k], lo_a[k], hi_a[k])}, LD-gap drop "
          f"{fmt_ci(gap_drop[k], lo_g[k], hi_g[k], 2)} (rank {int((gap_drop > gap_drop[k]).sum()) + 1}/1024)")
rho_ie_gap = stats.spearmanr(mean_ie.ravel(), gap_drop)[0]
rho_ie_auc = stats.spearmanr(mean_ie.ravel(), auc_drop)[0]
print(f"Spearman(patching mean IE, knockout LD-gap drop) = {rho_ie_gap:.2f}; (IE, AUC drop) = {rho_ie_auc:.2f}")

KO = {"n_per_class": N_KO, "sample": "held-out manuscript test set" if N_KO == 100 else "held-out rerun test set",
      "baseline_auc": A0, "baseline_ld_gap": g_all[0],
      "auc_drop": auc_drop.reshape(NL, NH), "auc_drop_ci95": [lo_a.reshape(NL, NH), hi_a.reshape(NL, NH)],
      "auc_drop_q_bh": q_a.reshape(NL, NH), "ld_gap_drop": gap_drop.reshape(NL, NH),
      "ld_gap_drop_ci95": [lo_g.reshape(NL, NH), hi_g.reshape(NL, NH)], "ld_gap_drop_q_bh": q_g.reshape(NL, NH),
      "top_auc_head": hn(top), "top_gap_head": hn(topg),
      "n_sig_auc": int((q_a < 0.05).sum()), "n_sig_gap": int((q_g < 0.05).sum()),
      "power": {"median_paired_se_auc_drop": np.median(se_a), "mde_80": mde,
                "max_upper_ci_auc_drop": hi_a.max(), "hanley_mcneil_mde_80": mde_hm},
      "spearman_patchIE_vs_gapdrop": rho_ie_gap, "spearman_patchIE_vs_aucdrop": rho_ie_auc}
save_json(f"head_knockout_fp16_heldout_n{N_KO}.json", KO)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for a_, M, ttl, lab in ((ax[0], auc_drop, "a  AUC drop", "baseline − knockout AUC"),
                        (ax[1], gap_drop, "b  LD-gap drop", "baseline − knockout LD gap (logits)")):
    v = np.abs(M).max()
    im = a_.imshow(M.reshape(NL, NH), cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
    a_.set_xlabel("head"); a_.set_ylabel("layer"); a_.set_title(ttl, loc="left")
    fig.colorbar(im, ax=a_, fraction=0.046, label=lab)
fig.suptitle(f"Single-head knockout, fp16, held-out n={N_KO}/class", x=0.01, ha="left", fontsize=9)
fig.tight_layout(); fig.savefig(FIGS / "head_knockout_fp16.png", dpi=200); fig.savefig(FIGS / "head_knockout_fp16.pdf")
plt.show()

## 10. Joint top-k heads (A1)
**Joint patching** (60 primary pairs): restore the clean `z` of the top-k patching heads *simultaneously*.
Head rankings: (i) *full* — ranked on all 60 pairs (circular, for comparability with the single-head table);
(ii) *cross-fit* — ranked on one random half of the pairs and evaluated on the other half (honest estimate).
**Joint knockout** (held-out test set from §9): zero the same top-k heads simultaneously.
Controls: 10 **layer-matched** random head sets per k (same layers, different heads).
Sanity check: joint patching with k=1 must reproduce the single-head IE stored in the rerun.

In [ ]:
pair_rows = list(csv.DictReader(open(RERUN / "liu_headtohead_metallama_v2_patching_pairs.csv")))
look_s = {h16(t): t for t in SUI[300:6300]}; look_n = {h16(t): t for t in NON[300:6300]}
PAIRS = [(look_s[r["hash_suicide"]], look_n[r["hash_non_suicide"]], r) for r in pair_rows]
PAIRS_P = [p for p, keep in zip(PAIRS, PRIM) if keep]
assert len(PAIRS_P) == nP
KS = [1, 3, 5, 10, 20, 40]; N_RAND = 10
rank_full = np.argsort(-mean_ie.ravel())
halves = np.random.default_rng(SEED + 21).permutation(nP); HA, HB = set(halves[: nP // 2]), set(halves[nP // 2:])
rank_A = np.argsort(-IEp[sorted(HA)].mean(0).ravel()); rank_B = np.argsort(-IEp[sorted(HB)].mean(0).ravel())
rr = np.random.default_rng(SEED + 23)

def layer_matched(heads):
    chosen = set(int(x) for x in heads); out = []
    for k in heads:
        L = int(k) // NH
        opts = [L * NH + h for h in range(NH) if L * NH + h not in chosen]
        c = int(rr.choice(opts)); chosen.add(c); out.append(c)
    return out

RAND_SETS = {k: [layer_matched(rank_full[:k]) for _ in range(N_RAND)] for k in KS}

def build_masks(head_sets):
    """Per layer: bool mask [B,1,NH*HD] marking the head slices to patch/zero in each row."""
    masks = {}
    for r, hs in enumerate(head_sets):
        for k in hs:
            L, h = divmod(int(k), NH)
            if L not in masks: masks[L] = torch.zeros(len(head_sets), 1, NH * HD, dtype=torch.bool, device=DEV)
            masks[L][r, :, h * HD:(h + 1) * HD] = True
    return masks

@torch.inference_mode()
def run_sets(ids, head_sets, clean_z=None, chunk=16):
    """LD for each head set (patch from clean_z if given, else zero) + one unpatched row per chunk."""
    out = []
    for c0 in range(0, len(head_sets), chunk):
        sets = head_sets[c0:c0 + chunk] + [[]]
        masks = build_masks(sets); hooks = []
        for L, M in masks.items():
            def pre(module, inp, M=M, L=L):
                x = inp[0]
                fill = clean_z[L].to(x.dtype)[None] if clean_z is not None else torch.zeros((), dtype=x.dtype, device=x.device)
                return (torch.where(M, fill, x),) + tuple(inp[1:])
            hooks.append(LAYERS[L].self_attn.o_proj.register_forward_pre_hook(pre))
        try:
            ld = ld_batch(ids.repeat(len(sets), 1))
        finally:
            for x in hooks: x.remove()
        out.append((ld[:-1], ld[-1]))
    return np.concatenate([o[0] for o in out]), np.mean([o[1] for o in out])

@torch.inference_mode()
def cache_z(ids):
    store = {}
    def mk(L):
        def pre(m, inp): store[L] = inp[0][0].detach().clone()
        return pre
    hs = [LAYERS[L].self_attn.o_proj.register_forward_pre_hook(mk(L)) for L in range(NL)]
    try:
        ld = ld_batch(ids)[0]
    finally:
        for x in hs: x.remove()
    return ld, store

# --- joint patching ---
JP = {"full": np.zeros((nP, len(KS))), "crossfit": np.zeros((nP, len(KS))),
      "random": np.zeros((nP, len(KS), N_RAND)), "sum_single_full": np.zeros((nP, len(KS)))}
t0 = time.time()
for j, (ts, tn, row) in enumerate(PAIRS_P):
    ids_s, _ = chat_ids(ts); ids_n, _ = chat_ids(tn)
    ld_c, z = cache_z(ids_s)
    rk = rank_B if j in HA else rank_A
    sets = ([list(rank_full[:k]) for k in KS] + [list(rk[:k]) for k in KS]
            + [s for k in KS for s in RAND_SETS[k]])
    ld, ld_corr = run_sets(ids_n, sets, clean_z=z)
    gap = ld_c - ld_corr
    ie = (ld - ld_corr) / gap
    JP["full"][j] = ie[:len(KS)]; JP["crossfit"][j] = ie[len(KS):2 * len(KS)]
    JP["random"][j] = ie[2 * len(KS):].reshape(len(KS), N_RAND)
    JP["sum_single_full"][j] = [IEp[j].ravel()[rank_full[:k]].sum() for k in KS]
    if j == 0:
        print(f"check pair 0: LD clean {ld_c:.2f} (rerun {float(row['ld_clean']):.2f}), corrupt {ld_corr:.2f} "
              f"(rerun {float(row['ld_corrupt']):.2f}); k=1 joint IE {ie[0]:.3f} vs stored single IE "
              f"{IEp[0].ravel()[rank_full[0]]:.3f}")
    if (j + 1) % 10 == 0: print(f"  {j + 1}/{nP} pairs ({time.time() - t0:.0f}s)")

def pair_ci(x):
    b = np.random.default_rng(SEED + 31).integers(0, len(x), (2000, len(x)))
    return x.mean(0), *ci(np.stack([x[i].mean(0) for i in b]))

print("\nJoint patching IE (mean over 60 pairs, 95% CI):")
print(f"{'k':>3}  {'top (full rank)':>24}  {'top (cross-fit)':>24}  {'layer-matched random':>22}  {'sum of single IEs':>18}")
JP_SUM = {}
mf, lf, hf = pair_ci(JP["full"]); mc, lc, hc = pair_ci(JP["crossfit"])
mr = JP["random"].mean(0)
ms = JP["sum_single_full"].mean(0)
for i, k in enumerate(KS):
    print(f"{k:3d}  {fmt_ci(mf[i], lf[i], hf[i]):>24}  {fmt_ci(mc[i], lc[i], hc[i]):>24}  "
          f"{mr[i].mean():.3f} [{mr[i].min():.3f}, {mr[i].max():.3f}]  {ms[i]:18.3f}")
    JP_SUM[k] = {"top_full": [mf[i], lf[i], hf[i]], "top_crossfit": [mc[i], lc[i], hc[i]],
                 "random_mean": mr[i].mean(), "random_range": [mr[i].min(), mr[i].max()],
                 "sum_of_single_ie": ms[i]}

# --- joint knockout on the held-out test posts used in §9 ---
JK_sets = [list(rank_full[:k]) for k in KS] + [s for k in KS for s in RAND_SETS[k]]
JK = np.zeros((len(ko_idx), len(JK_sets))); JK_base = np.zeros(len(ko_idx))
t0 = time.time()
for j, i in enumerate(ko_idx):
    JK[j], JK_base[j] = run_sets(TEST_IDS[i], JK_sets)
    if (j + 1) % 50 == 0: print(f"  knockout {j + 1}/{len(ko_idx)} ({time.time() - t0:.0f}s)")
S2 = np.concatenate([JK_base[:, None], JK], 1)
a2, g2 = auc_cols(Y_KO, S2), gap_cols(Y_KO, S2)
bt2 = np.array([np.r_[auc_cols(Y_KO[ix], S2[ix]), gap_cols(Y_KO[ix], S2[ix])] for ix in strat_boot(Y_KO, 1000)])
nC = S2.shape[1]
print(f"\nJoint knockout (held-out n={N_KO}/class): baseline AUC {a2[0]:.3f}, LD gap {g2[0]:.2f}")
JK_SUM = {}
for i, k in enumerate(KS):
    c = 1 + i; rnd = 1 + len(KS) + i * N_RAND + np.arange(N_RAND)
    da = bt2[:, 0] - bt2[:, c]; dg = bt2[:, nC] - bt2[:, nC + c]
    print(f"  top-{k:<2d}: AUC {a2[c]:.3f} (drop {fmt_ci(a2[0] - a2[c], *ci(da))}), LD gap {g2[c]:.2f} "
          f"(drop {fmt_ci(g2[0] - g2[c], *ci(dg), 2)});  random: AUC {a2[rnd].mean():.3f} "
          f"[{a2[rnd].min():.3f}, {a2[rnd].max():.3f}], gap {g2[rnd].mean():.2f}")
    JK_SUM[k] = {"auc": a2[c], "auc_drop_ci95": ci(da), "ld_gap": g2[c], "ld_gap_drop_ci95": ci(dg),
                 "random_auc_mean": a2[rnd].mean(), "random_auc_range": [a2[rnd].min(), a2[rnd].max()],
                 "random_gap_mean": g2[rnd].mean()}
save_json("joint_topk_heads.json", {"ks": KS, "n_random": N_RAND, "heads_full_rank": [hn(int(k)) for k in rank_full[:max(KS)]],
                                    "joint_patching": JP_SUM, "joint_knockout": JK_SUM,
                                    "baseline_auc": a2[0], "baseline_gap": g2[0]})
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
ax[0].plot(KS, mf, "o-", c="#b2182b", label="top-k (full rank)")
ax[0].plot(KS, mc, "s--", c="#ef8a62", label="top-k (cross-fit)")
ax[0].fill_between(KS, mr.min(1), mr.max(1), color="0.8", label="layer-matched random")
ax[0].plot(KS, ms, ":", c="0.4", label="sum of single-head IEs")
ax[0].set_xscale("log"); ax[0].set_xticks(KS, KS); ax[0].set_xlabel("k heads"); ax[0].set_ylabel("joint IE")
ax[0].axhline(1, c="0.6", lw=0.6); ax[0].set_title("a  Joint patching", loc="left"); ax[0].legend(frameon=False, fontsize=7)
rnd_auc = np.array([[a2[1 + len(KS) + i * N_RAND + r] for r in range(N_RAND)] for i in range(len(KS))])
ax[1].plot(KS, a2[1:1 + len(KS)], "o-", c="#2166ac", label="top-k knocked out")
ax[1].fill_between(KS, rnd_auc.min(1), rnd_auc.max(1), color="0.8", label="layer-matched random")
ax[1].axhline(a2[0], c="0.4", ls=":", label="baseline")
ax[1].set_xscale("log"); ax[1].set_xticks(KS, KS); ax[1].set_xlabel("k heads"); ax[1].set_ylabel("AUC")
ax[1].set_title("b  Joint knockout", loc="left"); ax[1].legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(FIGS / "joint_topk_heads.png", dpi=200); fig.savefig(FIGS / "joint_topk_heads.pdf")
plt.show()

## 11. Layer-wise attention/MLP patching, fp16, unclipped (A3)
Replaces `sc2_patching.py` (4-bit, 19 padded pairs from the training subset, IE clipped to ±1). Here: the same
60 held-out, exact-length-matched primary pairs as the head patching (no padding needed), signed IE, plus the
clipped mean only as a sensitivity check. One batched forward per pair per component: row L restores layer L.

In [ ]:
@torch.inference_mode()
def layer_patch(ids_c, ids_n):
    cache = {}
    def mk(key):
        def hook(m, inp, out): cache[key] = out[0].detach().clone()
        return hook
    hs = [LAYERS[L].self_attn.o_proj.register_forward_hook(mk(("attn", L))) for L in range(NL)]
    hs += [LAYERS[L].mlp.down_proj.register_forward_hook(mk(("mlp", L))) for L in range(NL)]
    try:
        ld_c = ld_batch(ids_c)[0]
    finally:
        for x in hs: x.remove()
    res = {}
    for comp in ("attn", "mlp"):
        hooks = []
        for L in range(NL):
            mod = LAYERS[L].self_attn.o_proj if comp == "attn" else LAYERS[L].mlp.down_proj
            def hook(m, inp, out, L=L, comp=comp):
                out = out.clone(); out[L] = cache[(comp, L)].to(out.dtype); return out
            hooks.append(mod.register_forward_hook(hook))
        try:
            ld = ld_batch(ids_n.repeat(NL + 1, 1))
        finally:
            for x in hooks: x.remove()
        res[comp] = ld[:NL]; res["corrupt_" + comp] = ld[NL]
    return ld_c, res

LP = {"attn": np.zeros((nP, NL)), "mlp": np.zeros((nP, NL))}
t0 = time.time()
for j, (ts, tn, _) in enumerate(PAIRS_P):
    ld_c, r = layer_patch(chat_ids(ts)[0], chat_ids(tn)[0])
    for comp in ("attn", "mlp"):
        LP[comp][j] = (r[comp] - r["corrupt_" + comp]) / (ld_c - r["corrupt_" + comp])
print(f"done ({time.time() - t0:.0f}s)")
LAYERPATCH = {}
fig, ax = plt.subplots(figsize=(7, 3.4))
for comp, col in (("attn", "#b2182b"), ("mlp", "#2166ac")):
    m, lo, hi = pair_ci(LP[comp]); mclip = np.clip(LP[comp], -1, 1).mean(0)
    LAYERPATCH[comp] = {"mean_ie": m, "ci95": [lo, hi], "clipped_mean_ie_sensitivity": mclip,
                        "peak_layer": int(np.argmax(m)), "peak_ie": m.max()}
    print(f"{comp}: peak L{int(np.argmax(m))} IE {fmt_ci(m.max(), lo[np.argmax(m)], hi[np.argmax(m)])}; "
          f"clipped peak L{int(np.argmax(mclip))} {mclip.max():.3f}; top-3 layers {list(np.argsort(-m)[:3])}")
    ax.plot(range(NL), m, "o-", ms=3, c=col, label="attention" if comp == "attn" else "MLP")
    ax.fill_between(range(NL), lo, hi, color=col, alpha=0.18, lw=0)
ax.axhline(0, c="0.6", lw=0.7); ax.set_xlabel("layer"); ax.set_ylabel("IE (signed, unclipped)")
ax.set_title(f"Layer-wise denoising patching, fp16, {nP} held-out length-matched pairs", loc="left", fontsize=9)
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(FIGS / "patching_ie_fp16.png", dpi=200); fig.savefig(FIGS / "patching_ie_fp16.pdf"); plt.show()
save_json("layer_patching_fp16.json", {"n_pairs": nP, "pairs": "rerun primary pairs (held out, exact length match)",
                                       **LAYERPATCH})

## 12. Transfer to SDCNL / SWMH in fp16, with and without overlapping posts
Reproduces `sc2_crossablation.py` (per-layer 232K directions ablated at **all** layers; 80/class) and the
decoding transfer of `sc2_generalization.py` (layer-16 projection; 120/class), in fp16, then recomputes both
after dropping any evaluation post flagged in §6 as an exact or near duplicate of a 232K post.
The manuscript text currently quotes the **4-bit** cross-ablation numbers (0.947→0.632, 0.932→0.631);
use the fp16 numbers printed here.

In [ ]:
DIRS = H_TR[:100].mean(0) - H_TR[100:].mean(0)
DIRS /= np.linalg.norm(DIRS, axis=1, keepdims=True) + 1e-8                      # [NL+1, DM]
_r = copy.deepcopy(RNG_AFTER_EXT)
RANDS = _r.standard_normal(DIRS.shape); RANDS /= np.linalg.norm(RANDS, axis=1, keepdims=True)
flag = globals().get("FLAGGED", set())
TRANSFER = {}
for name in ("SDCNL", "SWMH"):
    s, n = EXT[name][0][:120], EXT[name][1][:120]
    texts = s + n; y = np.array([1] * len(s) + [0] * len(n))
    ids = [chat_ids(t)[0] for t in texts]
    Hx = np.stack([hidden_states(i)[LAYER] for i in ids])
    proj = Hx @ DIRS[LAYER]
    lds = score(ids)
    sub80 = np.r_[np.arange(80), 120 + np.arange(80)]
    TRANSFER[name] = {"proj": proj, "ld": lds, "y": y, "ids": ids, "texts": texts, "sub80": sub80}

def score_perlayer(ids_list, dirs):
    """Ablate dirs[i+1] at decoder layer i, for every layer (sc2_crossablation.py)."""
    vs = [torch.tensor(dirs[i + 1], device=DEV, dtype=torch.float16) for i in range(NL)]
    hs = [LAYERS[i].register_forward_hook(make_ablate_hook(vs[i])) for i in range(NL)]
    try:
        return np.array([ld_batch(i)[0] for i in ids_list])
    finally:
        for x in hs: x.remove()

TRANSFER_OUT = {}
for name, T in TRANSFER.items():
    sub = T["sub80"]; ids80 = [T["ids"][i] for i in sub]; y80 = T["y"][sub]
    base80 = T["ld"][sub]; abl80 = score_perlayer(ids80, DIRS); rnd80 = score_perlayer(ids80, RANDS)
    keep120 = np.array([h16(t) not in flag for t in T["texts"]]); keep80 = keep120[sub]
    res = {}
    for tag, k120, k80 in (("all", np.ones(240, bool), np.ones(160, bool)), ("excluding_overlap", keep120, keep80)):
        yk = T["y"][k120]; S3 = np.stack([base80, abl80, rnd80], 1)[k80]; y3 = y80[k80]
        a3 = auc_cols(y3, S3); b3 = np.array([auc_cols(y3[ix], S3[ix]) for ix in strat_boot(y3, 2000)])
        dec = auc_cols(yk, np.stack([T["proj"][k120], T["ld"][k120]], 1))
        res[tag] = {"n_per_class_decoding": [int(yk.sum()), int((1 - yk).sum())],
                    "decoding_transfer_auc": dec[0], "behavioral_auc_120": dec[1],
                    "n_per_class_causal": [int(y3.sum()), int((1 - y3).sum())],
                    "causal": {"baseline": a3[0], "ablate_232K_dirs": a3[1], "random": a3[2],
                               "drop_ci95": ci(b3[:, 0] - b3[:, 1]), "random_drop_ci95": ci(b3[:, 0] - b3[:, 2])}}
        print(f"{name} [{tag}] decoding transfer AUC {dec[0]:.3f} (n={int(yk.sum())}+{int((1 - yk).sum())}); "
              f"causal: baseline {a3[0]:.3f} -> ablated {a3[1]:.3f} (drop {fmt_ci(a3[0] - a3[1], *ci(b3[:, 0] - b3[:, 1]))}),"
              f" random {a3[2]:.3f}")
    old = json.load(open(OLD / "crossablation.json"))["results"][name]
    res["old_fp16_crossablation_json"] = {c: old[c]["auc"] for c in old}
    TRANSFER_OUT[name] = res
save_json("transfer_fp16.json", TRANSFER_OUT)

## 12b. Transfer after removing preprocessing-invariant duplicates
The §6 audit compares normalized text, but 232K is stop-word-stripped and lemmatized while the "SDCNL" copy
(HF `Ram07/Detection-for-Suicide`, a cleaned release of the same Kaggle corpus) and SWMH are raw, so §6 misses
re-preprocessed copies of the same post. Here a post is flagged when ≥90% of the word types of some 232K post
(≥15 types) occur in it. Flagged posts are dropped from the 500/class draw and the first 80 (causal) / 120
(decoding) clean posts per class are re-scored. Only §0–1, §7 and §12 need to have run first (§9–11 can be skipped).

In [ ]:
import collections as _co
_tok = lambda x: set(re.findall(r"[a-z0-9]+", (x or "").lower()))
_A = [_tok(t) for t in T07]
_df = _co.Counter(w for s in _A for w in s)
_idx = _co.defaultdict(list)
for i, s in enumerate(_A):
    if len(s) < 15: continue
    for w in sorted(s, key=_df.__getitem__)[:2]: _idx[w].append(i)
FIT = set(SUI_TR + NON_TR)

def match_232k(text, thr=0.9):
    T = _tok(text); cands = set()
    for w in T:
        L = _idx.get(w)
        if L and len(L) <= 200: cands.update(L)
    if not cands: return None
    c, i = max((len(_A[i] & T) / len(_A[i]), i) for i in cands)
    return i if c >= thr else None

DEDUP = {}
for name in ("SDCNL", "SWMH"):
    cls_texts, info = [], {}
    for cls, texts in zip(("suicide", "non_suicide"), EXT[name]):
        m = [match_232k(t) for t in texts]
        info[cls] = {"flagged_first80": sum(x is not None for x in m[:80]),
                     "flagged_first120": sum(x is not None for x in m[:120]),
                     "flagged_of_500": sum(x is not None for x in m),
                     "flagged_direction_fit_posts": sum(x is not None and T07[x] in FIT for x in m)}
        cls_texts.append([t for t, x in zip(texts, m) if x is None])
    s, n = cls_texts[0][:120], cls_texts[1][:120]
    print(name, info, "| clean posts available:", len(cls_texts[0]), len(cls_texts[1]))
    texts = s + n; y = np.array([1] * len(s) + [0] * len(n))
    ids = [chat_ids(t)[0] for t in texts]
    proj = np.stack([hidden_states(i)[LAYER] for i in ids]) @ DIRS[LAYER]
    lds = score(ids)
    sub = np.r_[np.arange(80), len(s) + np.arange(80)]
    ids80 = [ids[i] for i in sub]; y80 = y[sub]
    S3 = np.stack([lds[sub], score_perlayer(ids80, DIRS), score_perlayer(ids80, RANDS)], 1)
    a3 = auc_cols(y80, S3); b3 = np.array([auc_cols(y80[ix], S3[ix]) for ix in strat_boot(y80, 2000)])
    dec = auc_cols(y, np.stack([proj, lds], 1))
    DEDUP[name] = {"flags": info, "n_per_class_decoding": [len(s), len(n)], "decoding_transfer_auc": dec[0],
                   "behavioral_auc_120": dec[1], "n_per_class_causal": [80, 80],
                   "causal": {"baseline": a3[0], "ablate_232K_dirs": a3[1], "random": a3[2],
                              "drop_ci95": ci(b3[:, 0] - b3[:, 1]), "random_drop_ci95": ci(b3[:, 0] - b3[:, 2])}}
    print(f"{name} [deduplicated] decoding {dec[0]:.3f}; causal {a3[0]:.3f} -> {a3[1]:.3f} "
          f"(drop {fmt_ci(a3[0] - a3[1], *ci(b3[:, 0] - b3[:, 1]))}), random {a3[2]:.3f}")
save_json("transfer_dedup.json", DEDUP)

## 13. Summary
Collects the headline numbers from every section into `summary.json` and zips `revision_colab/` next to it.
Send me `Dataset/Outputs/revision_colab.zip` (or the `summary.json`) and I'll update the manuscript text, tables and captions.

In [ ]:
summary = {}
for f in sorted(OUT.glob("*.json")):
    if f.name == "summary.json": continue
    d = json.loads(f.read_text())
    summary[f.stem] = {k: v for k, v in d.items()
                       if k not in ("auc_drop", "auc_drop_ci95", "auc_drop_q_bh", "ld_gap_drop", "ld_gap_drop_ci95",
                                    "ld_gap_drop_q_bh", "commits", "flagged_eval_hashes")}
(OUT / "summary.json").write_text(json.dumps({"summary": summary, "env": ENV}, indent=2, default=_js))
import shutil
shutil.make_archive(str(BASE / "revision_colab"), "zip", OUT)
print("wrote", OUT / "summary.json", "and", BASE / "revision_colab.zip")